# MonteGoal — v2 Model: rolling attack/defense strength + TimeSeriesSplit

In [12]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
from scipy.stats import poisson

DATA_PATH = "/Users/pranayagrawal/Desktop/COLLEGE/MonteGoal/data/raw/results_fifa.csv"
ROLL_WINDOW = 10          # matches of recent form per team
HALF_LIFE_YEARS = 6       # recency loss-weighting
HOLDOUT_FRACTION = 0.15   # final untouched chronological holdout
N_CV_SPLITS = 5
MAX_GOALS = 10            # cap for the Poisson outcome-probability grid

In [23]:
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df["match_id"] = df.index
#df = df[df.home_score != df.away_score].reset_index(drop=True)#review later
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,winner,venue,home_team_elo,away_team_elo,match_id
2193,2012-10-14,Trinidad and Tobago,Anguilla,10.0,0.0,CFU Caribbean Cup qualification,Trinidad and Tobago,Neutral,1489.0,497.0,2193
3447,2014-06-19,Japan,Greece,0.0,0.0,FIFA World Cup,Draw,Neutral,1770.0,1840.0,3447
592,2010-11-17,Albania,North Macedonia,0.0,0.0,Friendly,Draw,Albania,1544.0,1546.0,592
3269,2014-03-07,Lesotho,Eswatini,0.0,1.0,Friendly,Eswatini,Lesotho,1253.0,1255.0,3269
8338,2021-06-04,Guatemala,Saint Vincent and the Grenadines,10.0,0.0,FIFA World Cup qualification,Guatemala,Guatemala,1510.0,1104.0,8338


##  Rolling attack/defense strength (opponent-Elo-weighted, leak-free)

In [24]:
home_persp = pd.DataFrame({
    "match_id": df.match_id, "date": df.date, "team": df.home_team, "opponent": df.away_team,
    "opponent_elo": df.away_team_elo, "goals_scored": df.home_score, "goals_conceded": df.away_score,
    "side": "home",
})
away_persp = pd.DataFrame({
    "match_id": df.match_id, "date": df.date, "team": df.away_team, "opponent": df.home_team,
    "opponent_elo": df.home_team_elo, "goals_scored": df.away_score, "goals_conceded": df.home_score,
    "side": "away",
})
long = pd.concat([home_persp, away_persp], ignore_index=True).sort_values(["team", "date"]).reset_index(drop=True)

GLOBAL_FALLBACK = pd.concat([df.home_score, df.away_score]).mean()

def weighted_rolling(group):
    n = len(group)
    gs, gc, w = group.goals_scored.to_numpy(), group.goals_conceded.to_numpy(), group.opponent_elo.to_numpy()
    attack, defense = np.empty(n), np.empty(n)
    for i in range(n):
        if i == 0:
            attack[i] = defense[i] = GLOBAL_FALLBACK
            continue
        lo = max(0, i - ROLL_WINDOW)
        pw = w[lo:i]
        attack[i] = (gs[lo:i] * pw).sum() / pw.sum()
        defense[i] = (gc[lo:i] * pw).sum() / pw.sum()
    group = group.copy()
    group["attack_strength"], group["defense_strength"] = attack, defense
    return group

long = long.groupby("team", group_keys=False).apply(weighted_rolling)

home_feats = long[long.side == "home"][["match_id", "attack_strength", "defense_strength"]]
home_feats.columns = ["match_id", "home_attack_strength", "home_defense_strength"]
away_feats = long[long.side == "away"][["match_id", "attack_strength", "defense_strength"]]
away_feats.columns = ["match_id", "away_attack_strength", "away_defense_strength"]

df = df.merge(home_feats, on="match_id").merge(away_feats, on="match_id")
assert df[["home_attack_strength","home_defense_strength","away_attack_strength","away_defense_strength"]].isnull().sum().sum() == 0
df[["date","home_team","away_team","home_attack_strength","home_defense_strength"]].tail()

/var/folders/dw/7xzr4yh10v30b1f2r37p6gm00000gn/T/ipykernel_22223/2744244939.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long = long.groupby("team", group_keys=False).apply(weighted_rolling)


,date,home_team,away_team,home_attack_strength,home_defense_strength
12860,2026-06-06,Argentina,Honduras,2.303497,0.439039
12861,2026-06-06,Albania,Luxembourg,0.973008,1.020146
12862,2026-06-06,Vanuatu,Fiji,1.242110,2.656540
12863,2026-06-06,Sierra Leone,Liberia,1.114378,1.247232
12864,2026-06-06,Estonia,Faroe Islands,0.857026,2.471475


## Remaining features

In [25]:
df["elo_diff"] = df.home_team_elo - df.away_team_elo

# venue == away_team never occurs in this dataset (checked directly) - it's a true/false
# "is this the home team's own country" flag, not a three-way -1/0/1 signal
df["is_home_advantage"] = (df.venue == df.home_team).astype(int)

df["tournament"] = df["tournament"].astype("category")
df["year"] = df.date.dt.year   # kept only for weighting/splitting, NOT used as a model feature

FEATURES = ["elo_diff", "is_home_advantage", "home_attack_strength", "home_defense_strength",
            "away_attack_strength", "away_defense_strength", "tournament"]
df.head()

,date,home_team,away_team,home_score,away_score,tournament,winner,venue,home_team_elo,away_team_elo,match_id,home_attack_strength,home_defense_strength,away_attack_strength,away_defense_strength,elo_diff,is_home_advantage,year
0,2010-01-02,Qatar,Mali,0.0,0.0,Friendly,Draw,Qatar,1523.0,1508.0,0,1.32122,1.32122,1.32122,1.32122,15.0,1,2010
1,2010-01-02,Syria,Zimbabwe,6.0,0.0,Friendly,Syria,Neutral,1536.0,1454.0,1,1.32122,1.32122,1.32122,1.32122,82.0,0,2010
2,2010-01-02,Yemen,Tajikistan,0.0,1.0,Friendly,Tajikistan,Yemen,1253.0,1254.0,2,1.32122,1.32122,1.32122,1.32122,-1.0,1,2010
3,2010-01-03,Angola,Gambia,1.0,1.0,Friendly,Draw,Neutral,1430.0,1423.0,3,1.32122,1.32122,1.32122,1.32122,7.0,0,2010
4,2010-01-04,Egypt,Mali,1.0,0.0,Friendly,Egypt,Neutral,1754.0,1508.0,4,1.32122,1.32122,0.00000,0.00000,246.0,0,2010


##  Split: final chronological holdout + `TimeSeriesSplit` for CV


In [26]:
split_idx = int(len(df) * (1 - HOLDOUT_FRACTION))
split_date = df.iloc[split_idx].date
train_pool = df[df.date < split_date].copy()
holdout = df[df.date >= split_date].copy()

print(f"Train pool: {len(train_pool):,} ({train_pool.date.min().date()} -> {train_pool.date.max().date()})")
print(f"Holdout:    {len(holdout):,} ({holdout.date.min().date()} -> {holdout.date.max().date()})")

DECAY_RATE = 1
def recency_weight(sub_df, ref_year=None):
    ref_year = ref_year or sub_df.year.max()
    return DECAY_RATE ** (ref_year - sub_df.year)

Train pool: 10,920 (2010-01-02 -> 2024-03-21)
Holdout:    1,945 (2024-03-22 -> 2026-06-06)


In [27]:
tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
X_pool = train_pool[FEATURES]

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_pool)):
    fold_train, fold_val = train_pool.iloc[tr_idx], train_pool.iloc[val_idx]
    w = recency_weight(fold_train)  # fold-relative reference year
    m = XGBRegressor(objective="count:poisson", n_estimators=300, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, enable_categorical=True, random_state=42)
    m.fit(fold_train[FEATURES], fold_train.home_score, sample_weight=w)
    preds = np.clip(m.predict(fold_val[FEATURES]), 1e-6, None)
    print(f"fold {fold}: train={len(fold_train):,} val={len(fold_val):,}  "
          f"MAE={mean_absolute_error(fold_val.home_score, preds):.3f}")

fold 0: train=1,820 val=1,820  MAE=1.007
fold 1: train=3,640 val=1,820  MAE=0.993
fold 2: train=5,460 val=1,820  MAE=0.951
fold 3: train=7,280 val=1,820  MAE=1.027
fold 4: train=9,100 val=1,820  MAE=0.961


##  Final fit on the full train pool, evaluate once on the holdout

In [28]:
w_final = recency_weight(train_pool)
models, preds_holdout = {}, {}

for target in ["home_score", "away_score"]:
    m = XGBRegressor(objective="count:poisson", n_estimators=300, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, enable_categorical=True, random_state=42)
    m.fit(train_pool[FEATURES], train_pool[target], sample_weight=w_final)
    models[target] = m
    preds_holdout[target] = np.clip(m.predict(holdout[FEATURES]), 1e-6, None)
    print(f"[{target}]  MAE={mean_absolute_error(holdout[target], preds_holdout[target]):.3f}  "
          f"Poisson deviance={mean_poisson_deviance(holdout[target], preds_holdout[target]):.3f}")

[home_score]  MAE=0.997  Poisson deviance=1.145
[away_score]  MAE=0.822  Poisson deviance=1.130


In [29]:
importances = dict(zip(FEATURES, models["home_score"].feature_importances_.round(3)))
for k, v in sorted(importances.items(), key=lambda x: -x[1]):
    print(f"{k:25s} {v}")

elo_diff                  0.5860000252723694
away_defense_strength     0.09700000286102295
tournament                0.07999999821186066
home_attack_strength      0.07500000298023224
is_home_advantage         0.0729999989271164
away_attack_strength      0.04500000178813934
home_defense_strength     0.04399999976158142


##  Exact H/D/A probabilities from the two Poisson λ's (no Monte Carlo yet)

Rather than an arbitrary margin threshold on the two predicted goal rates, sum the joint
probability grid exactly: `P(home=i, away=j) = Poisson(i; λ_home) × Poisson(j; λ_away)`, then sum
the lower triangle (home win), diagonal (draw), upper triangle (away win). This *is* what Monte
Carlo sampling would converge to given enough simulations — doing it in closed form is exact, not
an approximation, and is the right way to get outcome probabilities at this stage.

In [30]:
def match_probs(lam_home, lam_away, max_goals=MAX_GOALS):
    hp = poisson.pmf(np.arange(max_goals + 1), lam_home)
    ap = poisson.pmf(np.arange(max_goals + 1), lam_away)
    grid = np.outer(hp, ap)
    return np.tril(grid, -1).sum(), np.trace(grid), np.triu(grid, 1).sum()  # P(home), P(draw), P(away)

pred_outcome = np.array([
    ["H", "D", "A"][np.argmax(match_probs(lh, la))]
    for lh, la in zip(preds_holdout["home_score"], preds_holdout["away_score"])
])
actual_outcome = np.where(holdout.home_score > holdout.away_score, "H",
                    np.where(holdout.home_score < holdout.away_score, "A", "D"))

print(f"Holdout outcome accuracy (argmax): {(pred_outcome == actual_outcome).mean():.3%}")
print("Predicted:", pd.Series(pred_outcome).value_counts().to_dict())
print("Actual:   ", pd.Series(actual_outcome).value_counts().to_dict())

Holdout outcome accuracy (argmax): 61.954%
Predicted: {'H': 1301, 'A': 644}
Actual:    {'H': 917, 'A': 557, 'D': 471}


##  Inference: `predict_matchup(team_a, team_b, ...)`


In [31]:
latest_home = df.sort_values("date").groupby("home_team").tail(1)[
    ["home_team","home_attack_strength","home_defense_strength","home_team_elo"]]
latest_home.columns = ["team","attack_strength","defense_strength","elo"]

latest_away = df.sort_values("date").groupby("away_team").tail(1)[
    ["away_team","away_attack_strength","away_defense_strength","away_team_elo"]]
latest_away.columns = ["team","attack_strength","defense_strength","elo"]

# if a team's most recent appearance was as home, prefer that row (and vice versa) -
# concat + groupby.last() keeps whichever came later chronologically
snapshot = pd.concat([latest_home, latest_away]).sort_values("team").groupby("team").last()

def predict_matchup(team_a, team_b, a_is_home, tournament, snapshot=snapshot, models=models):
    a, b = snapshot.loc[team_a], snapshot.loc[team_b]
    row = pd.DataFrame([{
        "elo_diff": a.elo - b.elo,
        "is_home_advantage": int(a_is_home),
        "home_attack_strength": a.attack_strength, "home_defense_strength": a.defense_strength,
        "away_attack_strength": b.attack_strength, "away_defense_strength": b.defense_strength,
        "tournament": tournament,
    }])
    row["tournament"] = row["tournament"].astype(pd.CategoricalDtype(categories=df.tournament.cat.categories))
    lam_a, lam_b = models["home_score"].predict(row)[0], models["away_score"].predict(row)[0]
    p_a, p_draw, p_b = match_probs(lam_a, lam_b)
    return {"lambda_A": round(float(lam_a), 3), "lambda_B": round(float(lam_b), 3),
            "P(A win)": round(float(p_a), 3), "P(draw)": round(float(p_draw), 3), "P(B win)": round(float(p_b), 3)}

predict_matchup("Spain", "Argentina", False, "FIFA World Cup")

{'lambda_A': 1.237,
 'lambda_B': 1.202,
 'P(A win)': 0.372,
 'P(draw)': 0.274,
 'P(B win)': 0.354}

In [33]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix, accuracy_score

pred_outcome = np.where(preds_holdout['home_score'] > preds_holdout['away_score'], 'H', 'A')
actual_outcome = np.where(holdout.home_score > holdout.away_score, 'H', 'A')

print(classification_report(actual_outcome, pred_outcome, labels=['H', 'A']))
print("Accuracy:", accuracy_score(actual_outcome, pred_outcome))
print("F1 (macro):", f1_score(actual_outcome, pred_outcome, average='macro'))
print("Confusion matrix:\n", confusion_matrix(actual_outcome, pred_outcome, labels=['H', 'A']))

              precision    recall  f1-score   support

           H       0.63      0.90      0.74       917
           A       0.85      0.53      0.66      1028

    accuracy                           0.70      1945
   macro avg       0.74      0.72      0.70      1945
weighted avg       0.75      0.70      0.70      1945

Accuracy: 0.7048843187660668
F1 (macro): 0.6989534301776246
Confusion matrix:
 [[822  95]
 [479 549]]


In [34]:
TOLERANCE = 1  # how many goals off still counts as "correct"

pred_home = np.round(np.clip(models["home_score"].predict(holdout[FEATURES]), 0, None)).astype(int)
pred_away = np.round(np.clip(models["away_score"].predict(holdout[FEATURES]), 0, None)).astype(int)

actual_home = holdout.home_score.to_numpy()
actual_away = holdout.away_score.to_numpy()

# per-team goal accuracy
exact_home = pred_home == actual_home
exact_away = pred_away == actual_away
home_within_tol = np.abs(pred_home - actual_home) <= TOLERANCE
away_within_tol = np.abs(pred_away - actual_away) <= TOLERANCE

# whole-match metrics
exact_scoreline = exact_home & exact_away
both_within_tol = home_within_tol & away_within_tol

pred_gd = pred_home - pred_away
actual_gd = actual_home - actual_away
gd_within_tol = np.abs(pred_gd - actual_gd) <= TOLERANCE

print(f"Home goals - exact match accuracy:     {exact_home.mean():.3%}")
print(f"Away goals - exact match accuracy:     {exact_away.mean():.3%}")
print(f"Home goals - within +/-{TOLERANCE} accuracy:   {home_within_tol.mean():.3%}")
print(f"Away goals - within +/-{TOLERANCE} accuracy:   {away_within_tol.mean():.3%}")
print(f"Both goals within +/-{TOLERANCE} (per match): {both_within_tol.mean():.3%}")
print(f"Exact scoreline accuracy:              {exact_scoreline.mean():.3%}")
print(f"Goal difference within +/-{TOLERANCE}:         {gd_within_tol.mean():.3%}")

Home goals - exact match accuracy:     31.105%
Away goals - exact match accuracy:     38.458%
Home goals - within +/-1 accuracy:   78.869%
Away goals - within +/-1 accuracy:   87.095%
Both goals within +/-1 (per match): 67.712%
Exact scoreline accuracy:              11.054%
Goal difference within +/-1:         67.558%


In [40]:
FEATUREX = ["elo_diff", "is_home_advantage", "home_attack_strength", "home_defense_strength",
            "away_attack_strength", "away_defense_strength", "tournament","home_score","away_score","home_team","away_team","date"]
export_df = df[FEATUREX]
export_df.to_csv("montegoal_featureX.csv", index=False)


In [36]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
from sklearn.preprocessing import StandardScaler

# --- one-hot encode tournament (a linear model can't use raw categoricals like XGBoost could) ---
train_X = pd.get_dummies(train_pool[FEATURES], columns=["tournament"], drop_first=True)
holdout_X = pd.get_dummies(holdout[FEATURES], columns=["tournament"], drop_first=True)
holdout_X = holdout_X.reindex(columns=train_X.columns, fill_value=0)  # align columns

# scale numeric features - the GLM optimizer converges far more reliably on similar-scale inputs
scaler = StandardScaler()
train_X_scaled = pd.DataFrame(scaler.fit_transform(train_X), columns=train_X.columns, index=train_X.index)
holdout_X_scaled = pd.DataFrame(scaler.transform(holdout_X), columns=holdout_X.columns, index=holdout_X.index)

baseline_models = {}
for target in ["home_score", "away_score"]:
    m = PoissonRegressor(alpha=1e-3, max_iter=1000)
    m.fit(train_X_scaled, train_pool[target], sample_weight=w_final)  # same recency weights as before
    baseline_models[target] = m
    preds = np.clip(m.predict(holdout_X_scaled), 1e-6, None)
    print(f"[{target}]  MAE={mean_absolute_error(holdout[target], preds):.3f}  "
          f"Poisson deviance={mean_poisson_deviance(holdout[target], preds):.3f}")

[home_score]  MAE=1.001  Poisson deviance=1.145
[away_score]  MAE=0.821  Poisson deviance=1.135


In [42]:
import numpy as np

# 1. Get the actual integer target values
actual_home = holdout.home_score.to_numpy()
actual_away = holdout.away_score.to_numpy()

# 2. Predict using the baseline_models dictionary and round to nearest integer
pred_home = np.round(np.clip(baseline_models["home_score"].predict(holdout_X_scaled), 1e-6, None))
pred_away = np.round(np.clip(baseline_models["away_score"].predict(holdout_X_scaled), 1e-6, None))

TOLERANCE = 1

# 3. Calculate per-team goal accuracy
exact_home = pred_home == actual_home
exact_away = pred_away == actual_away
home_within_tol = np.abs(pred_home - actual_home) <= TOLERANCE
away_within_tol = np.abs(pred_away - actual_away) <= TOLERANCE

# 4. Calculate whole-match metrics
exact_scoreline = exact_home & exact_away
both_within_tol = home_within_tol & away_within_tol

pred_gd = pred_home - pred_away
actual_gd = actual_home - actual_away
gd_within_tol = np.abs(pred_gd - actual_gd) <= TOLERANCE

# 5. Print the baseline results
print(f"Baseline Home goals - exact match accuracy:   {exact_home.mean():.3%}")
print(f"Baseline Away goals - exact match accuracy:   {exact_away.mean():.3%}")
print(f"Baseline Home goals - within +/-{TOLERANCE} accuracy: {home_within_tol.mean():.3%}")
print(f"Baseline Away goals - within +/-{TOLERANCE} accuracy: {away_within_tol.mean():.3%}")
print(f"Baseline Both goals within +/-{TOLERANCE} (per match): {both_within_tol.mean():.3%}")
print(f"Baseline Exact scoreline accuracy:            {exact_scoreline.mean():.3%}")
print(f"Baseline Goal difference within +/-{TOLERANCE}:       {gd_within_tol.mean():.3%}")


Baseline Home goals - exact match accuracy:   30.797%
Baseline Away goals - exact match accuracy:   38.920%
Baseline Home goals - within +/-1 accuracy: 80.000%
Baseline Away goals - within +/-1 accuracy: 87.249%
Baseline Both goals within +/-1 (per match): 68.740%
Baseline Exact scoreline accuracy:            11.517%
Baseline Goal difference within +/-1:       67.558%


In [37]:
w_final = recency_weight(train_pool)
models, preds_holdout = {}, {}

for target in ["home_score", "away_score"]:
    m = XGBRegressor(objective="count:poisson", n_estimators=300, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8, enable_categorical=True, random_state=42)
    m.fit(train_pool[FEATURES], train_pool[target], sample_weight=w_final)
    models[target] = m
    preds_holdout[target] = np.clip(m.predict(holdout[FEATURES]), 1e-6, None)
    print(f"[{target}]  MAE={mean_absolute_error(holdout[target], preds_holdout[target]):.3f}  "
          f"Poisson deviance={mean_poisson_deviance(holdout[target], preds_holdout[target]):.3f}")

[home_score]  MAE=0.997  Poisson deviance=1.145
[away_score]  MAE=0.822  Poisson deviance=1.130


In [41]:
export_df.sample(10)

,elo_diff,is_home_advantage,home_attack_strength,home_defense_strength,away_attack_strength,away_defense_strength,tournament,home_score,away_score,home_team,away_team,date
7079,324.0,1,1.590435,0.690962,0.419337,2.442674,Friendly,3.0,0.0,Colombia,Panama,2019-06-03
5474,-319.0,0,1.099536,1.414896,0.867347,0.842952,African Cup of Nations,0.0,2.0,Guinea-Bissau,Burkina Faso,2017-01-22
4913,142.0,1,1.318014,1.174366,1.146922,1.597739,Friendly,3.0,1.0,Serbia,Israel,2016-05-31
12114,309.0,1,1.649682,0.889252,0.677321,0.981868,Friendly,4.0,0.0,Greece,Bulgaria,2025-06-10
5769,-201.0,0,1.007418,1.577622,0.641548,0.812363,COSAFA Cup,1.0,2.0,Botswana,Zambia,2017-07-01
6048,53.0,1,1.297307,1.262005,1.050537,1.988421,Friendly,1.0,0.0,Georgia,Cyprus,2017-11-10
11008,-412.0,0,0.751723,2.551854,1.539128,0.853427,FIFA World Cup qualification,0.0,3.0,Yemen,United Arab Emirates,2024-03-26
11406,-145.0,1,2.288754,0.714783,1.936687,0.285730,FIFA World Cup qualification,2.0,1.0,Colombia,Argentina,2024-09-10
6781,125.0,1,1.671167,1.740718,0.588794,1.514682,Friendly,2.0,0.0,Argentina,Mexico,2018-11-16
7100,-92.0,0,0.477108,1.026738,1.201475,0.808306,COSAFA Cup,2.0,2.0,Lesotho,Zimbabwe,2019-06-07


## Next steps (v3 candidates)
- Hyperparameter search over `max_depth` / `learning_rate` / `min_child_weight` via the
  `TimeSeriesSplit` CV above — the flat MAE vs v1 points here first
- Dixon-Coles adjustment if draw calibration still matters after switching to log loss / Brier
- Score outcome quality with log loss / Brier score, not accuracy, going forward
- `ROLL_WINDOW` is currently a fixed guess (10) — worth sweeping